# Technique 1 — Being Clear and Direct

The **first line** of a prompt is the most important part. Two principles:

- **Clear** — simple language, no ambiguity; lead with a straightforward statement of the task.
- **Direct** — give an *instruction*, not a question; open with an action verb (`Write`, `Create`, `Generate`).

| | Prompt |
|---|--------|
| ❌ vague | *"What should this person eat?"* |
| ✅ clear & direct | *"Generate a one-day meal plan for an athlete that meets their dietary restrictions."* |

That one line tells Claude the **action** (generate), the **artifact** (a meal plan), and the **constraints** (one day, athlete, dietary restrictions). The course reports the eval score jumping **2.32 → 3.92** from this change alone.

> Same shared harness, same `dataset.json` as the baseline — only `run_prompt`'s first line changes, so the score delta is attributable to the technique.

## Load the harness + existing dataset

In [1]:
import sys
import os
from dotenv import find_dotenv

REPO_ROOT = os.path.dirname(find_dotenv())
SECTION = os.path.join(REPO_ROOT, "01-building-with-the-claude-api", "03-prompt-engineering")
if SECTION not in sys.path:
    sys.path.insert(0, SECTION)

from prompt_evaluator import PromptEvaluator, add_user_message, chat

DATASET = os.path.join(SECTION, "dataset.json")
evaluator = PromptEvaluator(max_concurrent_tasks=3)
print("Harness loaded. Reusing dataset ->", DATASET)

Harness loaded. Reusing dataset -> c:\Development\anthropic-cert\01-building-with-the-claude-api\03-prompt-engineering\dataset.json


## Improved prompt — clear & direct opening line

The inputs are unchanged; only the opening line is rewritten from a vague question into a direct instruction.

In [2]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## Re-evaluate against the same dataset

Same `extra_criteria` as the baseline (they're treated as mandatory). Compare the average score against the baseline's ~2.3.

In [3]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file=DATASET,
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
    json_output_file=os.path.join(SECTION, "output-02-clear-and-direct.json"),
    html_output_file=os.path.join(SECTION, "output-02-clear-and-direct.html"),
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 8.666666666666666


## Takeaway

A direct, action-verb opening line meaningfully lifts the score for essentially zero extra prompt length. Claude does best when treated as a capable assistant given clear direction — not one left to guess intent. It's still far from great, though: the next technique, **being specific**, spells out exactly *what the output must contain*.